df.groupby('key')['column'].mean()              # split, apply, combine

df.groupby('key')['column'].agg(['count', 'mean'])   # several summaries at once

df.groupby('key').agg({'a': 'mean', 'b': 'count'})   # different summaries, different columns

df.sort_values('column', ascending=False).head(n)

In [1]:
import pandas as pd
import numpy as np

url = 'https://eds-217-essential-python.github.io/data/marine_microplastics.csv'
df = pd.read_csv(url)

df = df.dropna(subset=['Measurement'])
samples = df[df['Unit'] == 'pieces/m3'].copy()
samples['ocean'] = samples['Oceans'].str.replace(' Ocean', '')
positive = samples[samples['Measurement'] > 0].copy()

positive.shape

(7091, 23)

In [2]:
positive.head()

,OBJECTID,Oceans,Regions,SubRegions,Sampling Method,Measurement,Unit,Density Range,Density Class,Short Reference,...,Keywords,Accession Number,Accession Link,Latitude,Longitude,Date,GlobalID,x,y,ocean
0,10008,Atlantic Ocean,NaN,NaN,Grab sample,0.020000,pieces/m3,0.005-1,Medium,Barrows et al.2018,...,Adventure Scientist/Citizen Science,211009,https://www.ncei.noaa.gov/access/metadata/land...,-58.428300,-64.1640,2/3/2017 12:00:00 AM,1e5b8e71-037b-4887-a276-f1e4552acb1f,-64.1640,-58.428300,Atlantic
1,8680,Atlantic Ocean,NaN,NaN,Grab sample,0.008000,pieces/m3,0.005-1,Medium,Barrows et al.2018,...,Adventure Scientist/Citizen Science,211009,https://www.ncei.noaa.gov/access/metadata/land...,-51.308200,-60.5467,11/17/2013 12:00:00 AM,a40f7f7c-1025-4aac-ad16-ee4cba196870,-60.5467,-51.308200,Atlantic
2,13257,Pacific Ocean,NaN,NaN,Manta net,0.019886,pieces/m3,0.005-1,Medium,Faure et al.2015,...,Oceaneye Association; Citizen Science,276422,https://www.ncei.noaa.gov/access/metadata/land...,-51.826667,-72.5750,12/26/2015 12:00:00 AM,febf79b8-7e2c-46e6-bc15-e08492ec2029,-72.5750,-51.826667,Pacific
3,9676,Atlantic Ocean,NaN,NaN,Grab sample,0.018000,pieces/m3,0.005-1,Medium,Barrows et al.2018,...,Adventure Scientist/Citizen Science,211009,https://www.ncei.noaa.gov/access/metadata/land...,-31.696000,-48.5600,8/11/2015 12:00:00 AM,a77121b2-e113-444e-82d9-7af11d62fdd2,-48.5600,-31.696000,Atlantic
5,10672,Pacific Ocean,NaN,NaN,Manta net,0.013000,pieces/m3,0.005-1,Medium,Goldstein et al.2013,...,Great Pacific Garbage Patch/SEAPLEX,253448,https://www.ncei.noaa.gov/access/metadata/land...,0.500000,-95.3500,10/17/2006 12:00:00 AM,23effcdd-35b7-4e1e-adb4-390693a287d3,-95.3500,0.500000,Pacific


### Pat 1. All four oceans at once

In [9]:
positive.groupby('Oceans')['Measurement'].mean().sort_values(ascending = False)



Oceans
Pacific Ocean     595.199821
Atlantic Ocean    282.729748
Arctic Ocean        2.279608
Southern Ocean      0.075703
Name: Measurement, dtype: float64

Yesterday you computed the mean Measurement for the Atlantic and the Pacific with two filters. Now do all four oceans in one line, and rank the result from dirtiest to cleanest.


    - Loook up :)

In [11]:
positive.groupby('Oceans')['Measurement'].median().sort_values(ascending = False)

Oceans
Pacific Ocean     0.116665
Arctic Ocean      0.044000
Atlantic Ocean    0.019440
Southern Ocean    0.011500
Name: Measurement, dtype: float64

#### Do the same with the median instead of the mean. Does the order change? If so, which two oceans swap

    - Yes, the atlantic and arctic swap.

In [16]:
positive.groupby('Oceans')['Measurement'].agg(['mean', 'median', 'count'])

,mean,median,count
Oceans,,,
Arctic Ocean,2.279608,0.044000,58
Atlantic Ocean,282.729748,0.019440,6216
Pacific Ocean,595.199821,0.116665,799
Southern Ocean,0.075703,0.011500,18


How many samples went into each of those four numbers? (can/should be a single line of code!)


    - Look up :)

#### 4. In a markdown cell: two of the four oceans have numbers you should refuse to report. Name those two oceans and explain why in one sentence.

     - Arctic and Southern because the sample size is too small when you compare their samples to the Atlantic and Pacific oceans.

### Part 2. One Call Instead of all 4

In [17]:
positive.groupby('Oceans')['Measurement'].agg(['mean', 'median', 'count', 'max'])

,mean,median,count,max
Oceans,,,,
Arctic Ocean,2.279608,0.044000,58,63.000000
Atlantic Ocean,282.729748,0.019440,6216,110480.000000
Pacific Ocean,595.199821,0.116665,799,21156.558533
Southern Ocean,0.075703,0.011500,18,0.499663


#### 5. Replace questions 1 through 3 with a single .agg() call that reports the count, median, mean and max of Measurement for each ocean

    - look up :)

#### 6. Look at the Atlantic row’s median and maximum. In a markdown cell, explain what a single large number does to a mean computed from 6,216 values. Which of the two measures of central tendency (mean vs. median) would you send to the journalist?

    - I would send the median because it better represents the spread of data and has less of an influence from outliers.



In [18]:
positive.groupby('Oceans').agg({
    'Measurement' : 'mean',
    'Sampling Method': 'count',
    'Organization': 'nunique'
})

,Measurement,Sampling Method,Organization
Oceans,,,
Arctic Ocean,2.279608,58,3
Atlantic Ocean,282.729748,6216,20
Pacific Ocean,595.199821,799,9
Southern Ocean,0.075703,18,3


#### 7. Use the dictionary form of .agg(), grouped by ocean, to report the mean Measurement, the number of distinct Sampling Method values, and the number of distinct Organization values in each ocean.

    - look up :)



#### 8. In a markdown cell explain the value of asking for these non-measurement summary data. What might it mean if one ocean had been sampled by only one organization using a single method?

     - When there are multiple orgs recording data then it allows for more errors to occur. If it was only one org then the data recording will be more consistent and free of errors. 

### Part 3. Where our answers fall apart

In [27]:
pos = positive.groupby('Sampling Method')['Measurement'].agg(['median', 'count']).sort_values('median', ascending = False)

pos

,median,count
Sampling Method,,
Stainless steel spoon,21840.000000,50
Van Dorn sampler,2000.000000,84
Aluminum bucket,1710.000000,57
PVC cylinder,1410.437236,292
CTD rosette sampler,21.000000,4
Intake seawater pump,2.500000,115
Plankton net,0.840000,74
Hand picking,0.777778,12
Manta net,0.252943,884


#### 9. Group by Sampling Method instead of by ocean, and report the count and median of Measurement for each method. Rank by median, largest first.


    - look up :)

In [39]:
smallest = pos['median'].min()
largest = pos['median'].max()

np.log10(largest / smallest)

6.561101383649056

#### 10. Write down the largest and smallest medians in that table. How many orders of magnitude separate them? (np.log10 of the ratio will tell you, or count the zeros.)


    - There are 6.6 orders of magnitude that seperate them

#### 11. In a markdown cell, before you write any more code: if the oceans were not sampled with the same mix of methods, what might that imply regarding your answer from question 1? Two or three sentences

    - Based off these results it could mean that those oceans might not be the dirtiest or cleanest. 

In [44]:
positive.groupby(['ocean', 'Sampling Method']).count().reset_index()

,ocean,Sampling Method,OBJECTID,Oceans,Regions,SubRegions,Measurement,Unit,Density Range,Density Class,...,Organization,Keywords,Accession Number,Accession Link,Latitude,Longitude,Date,GlobalID,x,y
0,Arctic,CTD rosette sampler,4,4,0,0,4,4,4,4,...,4,4,4,4,4,4,4,4,4,4
1,Arctic,Grab sample,21,21,21,0,21,21,21,21,...,21,21,21,21,21,21,21,21,21,21
2,Arctic,Intake seawater pump,6,6,0,0,6,6,6,6,...,6,6,6,6,6,6,6,6,6,6
3,Arctic,Manta net,27,27,27,0,27,27,27,27,...,27,27,27,27,27,27,27,27,27,27
4,Atlantic,Aluminum bucket,57,57,0,0,57,57,57,57,...,57,57,57,57,57,57,57,57,57,57
5,Atlantic,Grab sample,748,748,117,15,748,748,748,748,...,748,748,748,748,748,748,748,748,748,748
6,Atlantic,Hand picking,12,12,12,0,12,12,12,12,...,12,0,12,12,12,12,12,12,12,12
7,Atlantic,Intake seawater pump,106,106,12,1,106,106,106,106,...,106,106,106,106,106,106,106,106,106,106
8,Atlantic,Manta net,631,631,399,321,631,631,631,631,...,631,631,631,631,631,631,631,631,631,631
9,Atlantic,Neuston net,4390,4390,1156,17,4390,4390,4390,4390,...,4390,4390,4390,4390,4390,4390,4390,4390,4390,4390


#### 12. Let’s see if we can determine the impact of measurement method. Group by both ocean and Sampling Method at once and count the samples. Then call .reset_index() on the result and look at the Atlantic and Pacific rows

     - I looked.

In [64]:
atl = positive[positive['ocean'] == "Atlantic"]
atl['Sampling Method'].value_counts()


Sampling Method
Neuston net              4390
Grab sample               748
Manta net                 631
Intake seawater pump      106
PVC cylinder              104
Van Dorn sampler           84
Aluminum bucket            57
Stainless steel spoon      50
Plankton net               34
Hand picking               12
Name: count, dtype: int64

In [63]:
pac = positive[positive['ocean'] == "Pacific"]
pac['Sampling Method'].value_counts()

Sampling Method
Manta net               226
PVC cylinder            188
Grab sample             166
Neuston net             158
Plankton net             40
AVANI net                18
Intake seawater pump      3
Name: count, dtype: int64

In [66]:
len(pac['Sampling Method'])
158/799

0.19774718397997496

#### 13. Here’s a simpler and more direct approach: run .value_counts() on Sampling Method for the Atlantic samples and again for the Pacific samples. Which method dominates the Atlantic? What fraction of the Pacific samples used it?

     - The neuston net dominates the pacific and the faction of pacific samples that used it was 0.19774718397997496. 

In [82]:
new = positive[positive['Sampling Method'] == 'Neuston net'].copy()

new.groupby('ocean')['Measurement'].agg(['median', 'count', 'mean'])

,median,count,mean
ocean,,,
Atlantic,0.015120,4390,0.085767
Pacific,0.030680,158,0.396195
Southern,0.306271,4,0.278664


#### 14. Now let’s control for these differences. Filter positive to the rows collected with a 'Neuston net', end the line with .copy(), then group that by ocean and report count, median and mean.

    - look up :)